# ScoutLab — Player Scouting Demo

This notebook demonstrates the full ScoutLab pipeline:
1. Load and preprocess player data
2. Find similar players
3. Cluster players into archetypes
4. Visualize with radar and scatter charts

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure scoutlab is importable (run from repo root)
import sys
sys.path.insert(0, '..')

from scoutlab.pipeline.scouting import ScoutingPipeline

## 1. Create Sample Data

In a real workflow, replace this with a StatsBomb / FBref / Wyscout export.

In [ ]:
np.random.seed(42)

players = [
    "Pedri", "Gavi", "Frenkie de Jong", "Bellingham", "Modric",
    "Camavinga", "Valverde", "Kimmich", "Goretzka", "Rodri",
    "Verratti", "Fabian Ruiz", "Zielinski", "Musiala", "Saka",
    "Mount", "Curtis Jones", "Guido Rodriguez", "Bissouma", "Ndombele",
]

df = pd.DataFrame({
    "player": players,
    "goals": np.random.randint(2, 18, len(players)),
    "assists": np.random.randint(3, 15, len(players)),
    "xG": np.round(np.random.uniform(1.5, 16.0, len(players)), 2),
    "xA": np.round(np.random.uniform(2.0, 12.0, len(players)), 2),
    "progressive_passes": np.random.randint(60, 180, len(players)),
    "key_passes": np.random.randint(20, 90, len(players)),
    "dribbles": np.random.randint(15, 80, len(players)),
    "tackles": np.random.randint(20, 80, len(players)),
    "interceptions": np.random.randint(10, 60, len(players)),
    "pressures": np.random.randint(100, 400, len(players)),
    "minutes_played": np.random.randint(1500, 3200, len(players)),
})

df.head()

## 2. Fit the Pipeline

In [ ]:
pipeline = ScoutingPipeline(
    df=df,
    player_col="player",
    n_clusters=4,
    min_minutes=1000,
)
pipeline.fit()
print(f"Players in dataset: {len(pipeline.players)}")

## 3. Find Similar Players

In [ ]:
similar = pipeline.find_similar("Pedri", top_n=5)
print("Players most similar to Pedri:")
similar

In [ ]:
score = pipeline.similarity_score("Pedri", "Gavi")
print(f"Similarity between Pedri and Gavi: {score:.4f}")

## 4. Player Clusters

In [ ]:
print("Pedri's archetype:", pipeline.get_cluster("Pedri"))
print()
pipeline.cluster_summary()

## 5. Radar Chart

In [ ]:
pipeline.radar_chart(
    "Pedri",
    compare_with=["Gavi", "Bellingham"],
    metrics=["goals", "assists", "progressive_passes", "dribbles", "tackles", "pressures"],
    title="Pedri vs Gavi vs Bellingham",
)

## 6. Scatter: Goals vs xG

In [ ]:
pipeline.scatter_plot(
    x="xG",
    y="goals",
    highlight=["Pedri", "Bellingham"],
    title="Goals vs xG",
)

## 7. PCA Cluster Scatter

In [ ]:
pipeline.pca_scatter()

## 8. Save the Pipeline

In [ ]:
pipeline.save("scouting_pipeline.joblib")
print("Pipeline saved.")